# Project 03 · Neural Sequence Laboratory
## Session 1 — Teach an RNN to invent baby names

A name is a short sequence with visible regularities: beginnings, endings, repeated letters, and character combinations. In this notebook you will train a **character-level recurrent neural network** to learn those regularities and generate names that were not in its training data.

Most engineering is already complete. You will implement only five ideas from Chapter 5:

1. shift a sequence into next-character inputs and targets;
2. implement the recurrent state update;
3. compute sequence cross-entropy while ignoring padding;
4. clip the global gradient norm; and
5. sample autoregressively to invent names.

> **Success condition:** all five checks pass, the loss falls, and your model prints new names at three temperatures.

### Runtime expectations

The default model is intentionally small. Training should take a few minutes on a laptop CPU and less on a GPU. The notebook fixes random seeds, uses a deterministic train/validation split, and supplies every plotting and evaluation utility.

In [ ]:
from __future__ import annotations

import math
import random
import time
import urllib.request
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 351
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"PyTorch {torch.__version__} · device={DEVICE}")

## 0 · Data and vocabulary — provided

The dataset is the 32,033-name list used in Andrej Karpathy's *makemore* tutorial. The next cell downloads it once and caches it locally as `babynames.txt`.

Source: [karpathy/makemore — names.txt](https://github.com/karpathy/makemore/blob/master/names.txt).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
DATA_PATH = Path("babynames.txt")

if not DATA_PATH.exists():
    print("Downloading baby names…")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

names = [line.strip().lower() for line in DATA_PATH.read_text().splitlines() if line.strip()]
assert len(names) == 32_033, f"Expected 32,033 names, found {len(names):,}"
assert all(name.isalpha() and name.islower() for name in names)

lengths = [len(name) for name in names]
print(f"names: {len(names):,}")
print(f"length: min={min(lengths)}, mean={sum(lengths)/len(lengths):.2f}, max={max(lengths)}")
print("first 12:", names[:12])

In [ ]:
plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=range(1, max(lengths) + 2), color="#0f8b8d", edgecolor="white")
plt.xlabel("characters per name")
plt.ylabel("number of names")
plt.title("Baby-name length distribution")
plt.show()

We reserve three special tokens:

- `<PAD>` fills unused positions in a batch;
- `<BOS>` marks the beginning of a name;
- `<EOS>` marks the end of a name.

All vocabulary construction and encoding helpers are provided.

In [ ]:
PAD_TOKEN, BOS_TOKEN, EOS_TOKEN = "<PAD>", "<BOS>", "<EOS>"
characters = sorted(set("".join(names)))
itos = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN] + characters
stoi = {token: index for index, token in enumerate(itos)}
PAD_ID, BOS_ID, EOS_ID = (stoi[PAD_TOKEN], stoi[BOS_TOKEN], stoi[EOS_TOKEN])
VOCAB_SIZE = len(itos)

def encode_characters(text: str) -> list[int]:
    return [stoi[character] for character in text]

def decode_characters(ids: list[int]) -> str:
    return "".join(itos[index] for index in ids if index >= 3)

print(f"vocabulary size: {VOCAB_SIZE}")
print(stoi)

## Question 1 · Create next-character examples

A language model receives a sequence and predicts the token one position ahead. For `amina`:

```text
input:   <BOS> a m i n a
target:  a     m i n a <EOS>
```

Complete `make_example`. The returned lists must have equal length. Do not add padding here—the supplied batch function handles it later.

In [ ]:
def make_example(name: str) -> tuple[list[int], list[int]]:
    """Return shifted input and target token IDs for one name."""
    character_ids = encode_characters(name)

    # TODO Q1 — add the boundary token on the correct side of each sequence.
    input_ids = ...
    target_ids = ...

    return input_ids, target_ids

In [ ]:
# Check Q1
x_test, y_test = make_example("amina")
expected_x = [BOS_ID] + encode_characters("amina")
expected_y = encode_characters("amina") + [EOS_ID]
assert x_test == expected_x, f"input mismatch: {x_test}"
assert y_test == expected_y, f"target mismatch: {y_test}"
assert len(x_test) == len(y_test) == 6
print("✓ Q1 passed: the sequences are shifted by one position.")

### Dataset, split, and padding — provided

The split is deterministic: 80% training, 10% validation, and 10% test. Padding is added only inside each batch, so short names do not pay for the longest name in the entire dataset.

In [ ]:
class NameDataset(Dataset):
    def __init__(self, items: list[str]):
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> tuple[list[int], list[int]]:
        return make_example(self.items[index])

def collate_names(batch: list[tuple[list[int], list[int]]]) -> tuple[torch.Tensor, torch.Tensor]:
    max_length = max(len(inputs) for inputs, _ in batch)
    inputs = torch.full((len(batch), max_length), PAD_ID, dtype=torch.long)
    targets = torch.full((len(batch), max_length), PAD_ID, dtype=torch.long)
    for row, (input_ids, target_ids) in enumerate(batch):
        inputs[row, : len(input_ids)] = torch.tensor(input_ids)
        targets[row, : len(target_ids)] = torch.tensor(target_ids)
    return inputs, targets

shuffled_names = names.copy()
random.Random(SEED).shuffle(shuffled_names)
n_train = int(0.8 * len(shuffled_names))
n_valid = int(0.1 * len(shuffled_names))
train_names = shuffled_names[:n_train]
valid_names = shuffled_names[n_train:n_train + n_valid]
test_names = shuffled_names[n_train + n_valid:]

BATCH_SIZE = 256
train_loader = DataLoader(NameDataset(train_names), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_names, generator=torch.Generator().manual_seed(SEED))
valid_loader = DataLoader(NameDataset(valid_names), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_names)
test_loader = DataLoader(NameDataset(test_names), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_names)

batch_x, batch_y = next(iter(train_loader))
print(f"split: {len(train_names):,} / {len(valid_names):,} / {len(test_names):,}")
print("batch shapes:", tuple(batch_x.shape), tuple(batch_y.shape))

## Question 2 · Implement one recurrent state update

At every position the same cell computes

$$h_t = \tanh(W_x e_t + W_h h_{t-1}).$$

The embedding lookup, output layer, hidden-state initialization, and time loop are supplied. Complete only the state update in `step`. Your implementation must work on an entire batch.

In [ ]:
class CharacterRNN(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 32, hidden_dim: int = 96):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.input_to_hidden = nn.Linear(embedding_dim, hidden_dim)
        self.hidden_to_hidden = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.hidden_to_logits = nn.Linear(hidden_dim, vocab_size)

    def initial_state(self, batch_size: int, device: torch.device) -> torch.Tensor:
        return torch.zeros(batch_size, self.hidden_dim, device=device)

    def step(self, token_ids: torch.Tensor, previous_hidden: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        embedded = self.embedding(token_ids)

        # TODO Q2 — implement h_t = tanh(W_x e_t + W_h h_{t-1}).
        hidden = ...

        logits = self.hidden_to_logits(hidden)
        return logits, hidden

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps = token_ids.shape
        hidden = self.initial_state(batch_size, token_ids.device)
        outputs = []
        for time_step in range(time_steps):
            logits, hidden = self.step(token_ids[:, time_step], hidden)
            outputs.append(logits)
        return torch.stack(outputs, dim=1)

model = CharacterRNN(VOCAB_SIZE).to(DEVICE)
print(f"parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

In [ ]:
# Check Q2
probe = torch.tensor([[BOS_ID, stoi["a"], stoi["m"]], [BOS_ID, stoi["z"], stoi["o"]]], device=DEVICE)
probe_logits = model(probe)
assert probe_logits.shape == (2, 3, VOCAB_SIZE)
assert torch.isfinite(probe_logits).all()
probe_logits.sum().backward()
assert model.hidden_to_hidden.weight.grad is not None
model.zero_grad(set_to_none=True)
print("✓ Q2 passed: the shared recurrent cell produces differentiable sequence logits.")

## Question 3 · Compute padded sequence loss

`CrossEntropyLoss` expects class scores shaped `[examples, classes]` and targets shaped `[examples]`. Our tensors are:

```text
logits   [batch, time, vocabulary]
targets  [batch, time]
```

Flatten batch and time together. Padding targets must contribute no loss.

In [ ]:
loss_function = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def sequence_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Mean next-character cross-entropy over non-padding positions."""
    # TODO Q3 — flatten the correct dimensions, then call loss_function.
    flat_logits = ...
    flat_targets = ...
    return ...

In [ ]:
# Check Q3: changing logits at a PAD position must not change the loss.
toy_logits = torch.zeros(1, 3, VOCAB_SIZE)
toy_targets = torch.tensor([[stoi["a"], EOS_ID, PAD_ID]])
loss_before = sequence_loss(toy_logits, toy_targets)
toy_logits[:, 2, :] = torch.randn(VOCAB_SIZE) * 100
loss_after = sequence_loss(toy_logits, toy_targets)
assert loss_before.ndim == 0 and torch.isfinite(loss_before)
assert torch.allclose(loss_before, loss_after), "PAD position affected the loss"
assert abs(loss_before.item() - math.log(VOCAB_SIZE)) < 1e-5
print("✓ Q3 passed: loss uses every real target and ignores padding.")

## Question 4 · Clip the global gradient norm

BPTT can produce an unstable gradient. Complete the one-line guardrail

$$\tilde g = \min\left(1, \frac{\tau}{\lVert g\rVert_2}\right)g.$$

PyTorch already implements this operation and returns the norm **before** clipping.

In [ ]:
def clip_gradients(model: nn.Module, max_norm: float) -> float:
    """Clip all parameter gradients by global norm and return the pre-clip norm."""
    # TODO Q4 — use torch.nn.utils.clip_grad_norm_.
    pre_clip_norm = ...
    return float(pre_clip_norm)

In [ ]:
# Check Q4 with a deliberately enormous gradient.
clip_probe = nn.Linear(3, 2)
for parameter in clip_probe.parameters():
    parameter.grad = torch.full_like(parameter, 100.0)
pre_clip = clip_gradients(clip_probe, max_norm=1.0)
post_clip = torch.sqrt(sum(parameter.grad.square().sum() for parameter in clip_probe.parameters())).item()
assert pre_clip > 1.0
assert post_clip <= 1.00001
print(f"✓ Q4 passed: global norm {pre_clip:.1f} → {post_clip:.1f}")

## 1 · Train the model — provided

You have now supplied every chapter-specific operation needed for training. The optimization loop, validation pass, timing, history, and plotting are provided below.

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    weighted_loss = 0.0
    token_count = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        logits = model(inputs)
        loss = sequence_loss(logits, targets)
        real_tokens = targets.ne(PAD_ID).sum().item()
        weighted_loss += loss.item() * real_tokens
        token_count += real_tokens
    return weighted_loss / token_count

def train_model(model: nn.Module, epochs: int = 15, learning_rate: float = 2e-3, max_norm: float = 1.0) -> dict[str, list[float]]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history = {"train": [], "validation": [], "max_pre_clip_norm": []}

    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        model.train()
        weighted_loss = 0.0
        token_count = 0
        largest_norm = 0.0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)
            loss = sequence_loss(logits, targets)
            loss.backward()
            largest_norm = max(largest_norm, clip_gradients(model, max_norm))
            optimizer.step()

            real_tokens = targets.ne(PAD_ID).sum().item()
            weighted_loss += loss.item() * real_tokens
            token_count += real_tokens

        train_loss = weighted_loss / token_count
        validation_loss = evaluate(model, valid_loader)
        history["train"].append(train_loss)
        history["validation"].append(validation_loss)
        history["max_pre_clip_norm"].append(largest_norm)
        elapsed = time.perf_counter() - started
        print(f"epoch {epoch:02d} · train {train_loss:.3f} · valid {validation_loss:.3f} · ppl {math.exp(validation_loss):.2f} · max ‖g‖ {largest_norm:.2f} · {elapsed:.1f}s")

    return history

In [ ]:
model = CharacterRNN(VOCAB_SIZE, embedding_dim=32, hidden_dim=96).to(DEVICE)
history = train_model(model, epochs=15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
epochs = range(1, len(history["train"]) + 1)
axes[0].plot(epochs, history["train"], marker="o", label="train")
axes[0].plot(epochs, history["validation"], marker="o", label="validation")
axes[0].set(xlabel="epoch", ylabel="cross-entropy", title="Learning curves")
axes[0].legend()
axes[1].plot(epochs, history["max_pre_clip_norm"], marker="o", color="#d95f43")
axes[1].axhline(1.0, linestyle="--", color="#17364a", label="clip threshold")
axes[1].set(xlabel="epoch", ylabel="largest pre-clip norm", title="Gradient diagnostic")
axes[1].legend()
plt.tight_layout()
plt.show()

test_loss = evaluate(model, test_loader)
print(f"held-out test loss: {test_loss:.3f}")
print(f"held-out character perplexity: {math.exp(test_loss):.2f}")

### Checkpoint interpretation

Before generating, answer briefly:

1. Did training and validation loss both decrease?
2. Did the pre-clipping norm ever exceed the threshold? Why can the plotted value exceed 1 even though gradients were clipped?
3. Character perplexity is the effective number of equally likely next characters. Is your model substantially better than a uniform distribution over the vocabulary?

## Question 5 · Generate a new name

Generation repeats one operation:

```text
previous character → recurrent step → next-character distribution → sample → feed back
```

Complete the sampling block. Divide logits by `temperature` **before** softmax, squeeze the batch dimension, and move the small probability vector to CPU before calling `torch.multinomial`. Stop when `<EOS>` is sampled or when `max_length` is reached. Never return `<PAD>` or `<BOS>` as visible characters.

In [ ]:
@torch.no_grad()
def generate_name(model: CharacterRNN, temperature: float = 0.8, max_length: int = 20, seed: int | None = None) -> str:
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    model.eval()
    generator = torch.Generator()
    if seed is not None:
        generator.manual_seed(seed)
    else:
        generator.seed()

    hidden = model.initial_state(batch_size=1, device=DEVICE)
    current = torch.tensor([BOS_ID], device=DEVICE)
    generated_ids: list[int] = []

    for _ in range(max_length):
        logits, hidden = model.step(current, hidden)

        # TODO Q5 — temperature, softmax, sampling, stopping, and feedback.
        probabilities = ...
        next_id = ...
        if ...:
            break
        if ...:
            generated_ids.append(next_id)
        current = ...

    return decode_characters(generated_ids)

In [ ]:
# Check Q5
sample_a = generate_name(model, temperature=0.8, seed=123)
sample_b = generate_name(model, temperature=0.8, seed=123)
assert sample_a == sample_b, "fixed seed should reproduce the same sample"
assert 0 <= len(sample_a) <= 20
assert all(character in characters for character in sample_a)
print(f"✓ Q5 passed: reproducible sample = {sample_a!r}")

## Success moment · Invent names

Run the next cell more than once. Low temperature favors familiar combinations; high temperature increases variety and mistakes. A generated name is marked `NEW` only if it is absent from the complete dataset.

In [ ]:
known_names = set(names)
for temperature in (0.55, 0.80, 1.15):
    print(f"\ntemperature = {temperature:.2f}")
    produced = []
    attempts = 0
    while len(produced) < 12 and attempts < 100:
        candidate = generate_name(model, temperature=temperature)
        attempts += 1
        if candidate and candidate not in produced:
            produced.append(candidate)
    for candidate in produced:
        status = "NEW" if candidate not in known_names else "seen"
        print(f"  {candidate:<18} {status}")

## Final reflection

Write 150–250 words addressing all four points:

1. Give two examples of character patterns the RNN appears to have learned.
2. Compare the three temperatures using evidence from your samples.
3. Identify one failure that suggests the vanilla RNN struggles with longer-range structure.
4. Predict one way an LSTM or GRU might change the results in Session 2.

### Optional checkpoint

Save the trained state if you want to reuse it later:

```python
torch.save(model.state_dict(), "names_rnn.pt")
```

Before submission, restart the kernel and run all cells from top to bottom. Every `✓ Q… passed` message must appear.